# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata and print overview
metadata = dataset.metadata
print("Dataset title:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))

## 2. Data Overview
Review available record sets, their fields, and their corresponding `@id`s.

**Note:** We use the official `@id` values as unique identifiers from the schema. All entity references (record sets and fields) use their `@id`s for programmatic consistency.

In [ ]:
# List all available record sets and their fields
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, 'id', '')}, type: {getattr(field, 'data_type', '')})")
    print("\n")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

> Note: We'll extract all record sets into DataFrames. Adjust field usage as desired by referencing the correct `@id` values.

In [ ]:
# Extract all record sets into DataFrames
dataframes = {}
for rs in dataset.record_sets():
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded record set {rs.name} (@id: {rs.id}): Shape {df.shape}")

# Pick the main table for demonstration (first discovered set)
main_record_set_id = None
main_df = None
for rs_id, df in dataframes.items():
    if main_record_set_id is None:
        main_record_set_id = rs_id
        main_df = df

if main_df is not None:
    print("\nColumns (@id):", main_df.columns.tolist())
    display(main_df.head())
else:
    print("No record sets with records found!")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps such as filtering, normalization, and grouping based on specific field values.

In this section, we will:
- Select a numeric field (by `@id`) for demonstration.
- Apply filtering and normalization.
- Group by another field if appropriate.

> **Replace** the example field `@id` values below with those listed in Section 2 for deeper exploration.

In [ ]:
import numpy as np

# Demo selection: use a numeric field found in the main DataFrame (adjust as per the printed columns above)
if main_df is not None and not main_df.empty:
    # Try to heuristically pick a numeric column for demo
    numeric_field_id = None
    for col in main_df.columns:
        col_values = main_df[col].values
        # Check if the column seems numeric (can coerce at least one value)
        try:
            float(main_df[col].dropna().astype(float).iloc[0])
            numeric_field_id = col
            break
        except (ValueError, IndexError, TypeError):
            continue
    if numeric_field_id:
        print(f"Using numeric field for analysis: {numeric_field_id}")
        # Coerce to numeric
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
        # Filter for demonstration
        threshold = np.nanpercentile(main_df[numeric_field_id].dropna(), 75)
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} above 75th percentile (≈ {threshold:.2f}):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a string/categorical field
        group_field_id = None
        for col in main_df.columns:
            if col != numeric_field_id and main_df[col].dtype == object:
                vals = main_df[col].dropna().unique()
                if 1 < len(vals) < len(main_df) // 2:
                    group_field_id = col
                    break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped (record count: {len(grouped_df)}) by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical field found for grouping.")
    else:
        print("No numeric field detected in the primary record set.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` and/or `seaborn`.

Example: Show distribution and group comparison for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, you have:
- Discovered and described the dataset structure using the Croissant schema and `mlcroissant`
- Loaded and previewed the available record sets and their fields (with `@id`s)
- Performed exploratory data filtering and normalization for numeric fields
- Visualized the main distributions and potential group differences

For deeper analysis, select specific record sets and fields relevant to your research using their `@id` from the overview section.